In [1]:
from langchain_groq import ChatGroq
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableLambda
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.document_loaders import WebBaseLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_community.document_loaders import PyPDFLoader


from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

C:\Users\chait\AppData\Local\Temp\ipykernel_6708\517166502.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
c:\Users\chait\OneDrive\Desktop\Mern_Projects\AI\ChatBot\ChatBot\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
doc = WebBaseLoader("https://www.vectorindia.org/").load()
doc1=PyPDFLoader("media_to_upload1707550335.pdf").load()

chunks = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=150
).split_documents(doc+doc1)

fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/YKGIYZ+Times-Roman', '/FontDescriptor': IndirectObject(1430, 0, 1674237536208), '/Type': '/Font', '/FirstChar': 32, '/LastChar': 116, '/Widths': [250, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 250, 333, 0, 0, 0, 500, 500, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 722, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 722, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 444, 0, 444, 0, 444, 0, 0, 500, 278, 0, 0, 0, 0, 500, 500, 500, 0, 333, 389, 278], '/Encoding': '/WinAnsiEncoding', '/Subtype': '/Type1'}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/YKGIYZ+Times-Roman', '/FontDescriptor': IndirectObject(1430, 0, 1674237536208), '/Type': '/Font', '/FirstChar': 32, '/LastChar': 116, '/Widths': [250, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 250, 333, 0, 0, 0, 500, 

In [3]:
vdb = Chroma.from_documents(
    documents=chunks
)

db = vdb.as_retriever(
    search_kwargs={"k": 5}
)

In [4]:
kws=BM25Retriever.from_documents(chunks)
kws.k=3
hybrid_retriever=EnsembleRetriever(retrievers=[db,kws],weights=[0.7,0.3])

In [5]:
cross_encoder = HuggingFaceCrossEncoder(
    model_name="cross-encoder/ms-marco-MiniLM-L-6-v2"
)

compressor = CrossEncoderReranker(
    model=cross_encoder,
    top_n=3
)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 10038.34it/s]


In [6]:
compression_retriever = ContextualCompressionRetriever(
    base_retriever=hybrid_retriever,
    base_compressor=compressor
)

In [7]:
rewrite_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a query rewriting assistant for a conversational RAG system.

Your task is to rewrite the user's latest query into a clear,
specific, and self-contained search query.

Use the conversation history to understand:

- What the user is referring to
- Pronouns such as "it", "this", "that", "they", etc.
- Previously mentioned topics
- Previously mentioned entities
- Constraints mentioned earlier

Rules:

1. Preserve the user's original intent.
2. Do not add requirements that the user did not ask for.
3. Resolve references using the conversation history.
4. Make the query self-contained.
5. Remove unnecessary conversational words.
6. Do NOT answer the question.
7. Do NOT explain your reasoning.
8. Output ONLY the rewritten query.
9. If the query is already clear, return it with minimal changes.
"""
    ),
    MessagesPlaceholder(variable_name="chat_history"),
    ("user", "{query}")
])

In [8]:
qa_prompt = ChatPromptTemplate.from_messages([
    (
       "system",
        """
You are a helpful and accurate university assistant.

Answer the user's question using the provided context.

Rules:

1. Use the retrieved context as the primary source.
2. Use chat history only to understand references.
3. Do not invent information.
4. Do not make unsupported assumptions.
5. If the answer cannot be found in the context, say:

"I don't have enough information in the provided context to answer this question."

6. Give a clear and concise answer.
7. If appropriate, provide steps or examples.
8. Resolve pronouns such as "it", "this", and "that" using
   the conversation history.
9. Do not mention internal implementation details unless the
   user specifically asks about them.
"""
    ),
    MessagesPlaceholder(variable_name="chat_history"),
    (
        "user",
        """Context:
{context}

Question:
{query}"""
    )
])

In [9]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    max_retries=2
)

In [10]:
rewriting_chain = (
    rewrite_prompt
    | llm
    | StrOutputParser()
)

In [11]:
def retrieve_docs(x):

    rewritten_query = rewriting_chain.invoke({
        "query": x["query"],
        "chat_history": x["chat_history"]
    })
    print("\nRewritten query:", rewritten_query)

    docs = compression_retriever.invoke(
        rewritten_query
    )

    return {
        "context":docs ,
        "query": x["query"],
        "chat_history": x["chat_history"]
    }

In [12]:
rag_chain = (
    RunnableLambda(retrieve_docs)
    | qa_prompt
    | llm
)

In [13]:
store = {}

def get_session_history(session_id):

    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()

    return store[session_id]

In [14]:
conversation = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="query",
    history_messages_key="chat_history"
)

c:\Users\chait\OneDrive\Desktop\Mern_Projects\AI\ChatBot\ChatBot\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [16]:
config = {
    "configurable": {
        "session_id": "user-1"
    }
}

while True:

    user_input = input("User: ")

    if user_input.lower() in ["exit", "quit"]:
        break

    res = conversation.invoke(
        {
            "query": user_input
        },
        
        config=config
    )

    print("Bot:", res.content)
    print("--" * 60)


Rewritten query: What is Python programming language?
Bot: **Python**  
Python is a popular, interpreted, object‑oriented, high‑level programming language created by Guido van Rossum and first released in 1991. It is designed to be easy and fun, with a beginner‑friendly reputation that has made it a common choice for web development, software development, mathematics, system scripting, and big‑data processing.
------------------------------------------------------------------------------------------------------------------------

Rewritten query: Vector Institute India location and contact details.
Bot: **Vector India Institute**

- India's first ISO 9001:2015 certified embedded‑systems training institute.  
- Over 20 years of experience in advanced embedded‑systems training and placement.  
- More than 1,600 placements and 759 hiring MNCs.  
- Campuses located in Hyderabad, Bangalore and Chennai.  
- Focuses on industry‑ready, application‑oriented training in C, Linux, microcontrolle

KeyboardInterrupt: Interrupted by user